# Getting Started with catsim

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/01_getting_started.ipynb)

**For:** researchers who know the basics of item response theory and are new to catsim. **Level:**
beginner.

You will build an item bank, generated or read from a file of calibrated parameters, see what its
items measure, give a computerized adaptive test to 500 simulated examinees, read the result's
metrics and tables, follow one examinee's test item by item, and repeat the run from its seed. The
[introduction](https://douglasrizzo.com.br/catsim/introduction.html) explains adaptive testing, and
[Terms and notation](https://douglasrizzo.com.br/catsim/terms-and-notation.html) the terms used
here.

To run the notebook outside this repository, install catsim with its plot extra, which adds
matplotlib: `pip install 'catsim[plot]'`.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy

from catsim import CatEngine, ItemBank, plot, simulate
from catsim.estimation import EAPEstimator
from catsim.initialization import FixedPointInitializer
from catsim.selection import MaxInfoSelector
from catsim.stopping import MaxLengthStopper, StandardErrorStopper, StopRule

## An item bank

An item bank holds the calibrated parameters of its items, one row per item: discrimination $a$,
difficulty $b$, guessing parameter $c$ and upper asymptote $d$. `ItemBank.generate` draws a bank of
300 items of the three-parameter logistic model (3PL), whose upper asymptotes are all 1, for the
simulation below. Every item has a name; a generated bank names its items after their rows.

The notebook draws random numbers for three things: the item bank, the examinees' true abilities,
and the run, which draws each examinee's responses. Each has a seed of its own, so every cell gives
the same numbers each time it runs, and changing one seed changes only what it draws.

In [ ]:
BANK_SEED = 1
POPULATION_SEED = 2
RUN_SEED = 3

item_bank = ItemBank.generate(300, model="3PL", rng=BANK_SEED)
print(item_bank)
print(item_bank.names[:3])
print(item_bank.items[:3].round(2))

A bank you calibrated is read from a CSV file with a header row: the columns `a`, `b`, `c` and `d`,
in any order, and an optional `name` column. Only `b` is required; a missing `a` is 1, a missing `c`
is 0 and a missing `d` is 1. The file below holds five 3PL items, so it has no `d` column.
`ItemBank(matrix, names=...)` builds the same bank from an array, and `item_bank.to_csv(path)` saves
a bank, such as the generated one, so that a study can be repeated.

catsim's logistic models have no scaling constant. If your calibration reports discriminations for a
model with $D = 1.7$, multiply them by 1.7. If it reports an intercept rather than a difficulty, as
mirt does by default, the difficulty is $b = -\text{intercept} / a$.

In [ ]:
csv_path = Path(tempfile.mkdtemp()) / "calibrated.csv"
csv_path.write_text("""\
name,a,b,c
Q1,0.8,-1.5,0.20
Q2,1.2,-0.5,0.15
Q3,1.9,0.0,0.25
Q4,1.0,0.7,0.10
Q5,1.5,1.6,0.20
""")
my_bank = ItemBank.from_csv(csv_path)
print(my_bank, my_bank.names)
print(my_bank.items)

## What an item measures

An item's characteristic curve gives the probability of a correct response at each ability, and its
information curve how much a response to it sharpens an ability estimate there. `plot.item_curve`
draws both for one item, given its index and its bank; `my_bank.index` finds the index of a name.
Below are Q1, easy and weakly discriminating, and Q3, of middle difficulty and the most
discriminating.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot.item_curve(my_bank.index("Q1"), my_bank, kind="both", ax=axes[0])
plot.item_curve(my_bank.index("Q3"), my_bank, kind="both", ax=axes[1])
plt.tight_layout()
plt.show()

Each characteristic curve (blue, left axis) rises from the guessing parameter, 0.2 and 0.25, toward
1, most steeply near the difficulty. Q3's larger discrimination makes its curve steeper and its
information (red, right axis) peak about five times higher, at about 0.56 against 0.11, but over a
narrower range of abilities. The dot marks each peak, a little above the difficulty: with guessing,
a correct response says less about an examinee of low ability. An adaptive test gives each item to
the examinees near its peak.

## A simulation

A test is an item bank and four strategies, which a `CatEngine` holds:

- an initializer gives the ability estimate before the first response, here 0;
- a selector chooses each item, here the most informative one at the current estimate;
- an estimator updates the estimate after each response, here the expected a posteriori (EAP)
  estimate, which starts from a standard normal prior;
- a stopper ends the test, here once the standard error of the estimate is at most `TARGET_SE`, or
  after `MAX_ITEMS` items, whichever comes first.

The length cap ends the test of an examinee whose target precision the bank cannot reach. `simulate`
gives the test to each of `N_EXAMINEES` examinees, whose true abilities are drawn from a standard
normal distribution, and returns a `SimulationResult`.

Each metric of a result is a value and its Monte Carlo standard error, `mc_se`: how much the value
would vary if the run were repeated with other seeds. The bias is the mean of the estimates' errors,
$\hat{\theta} - \theta$, and the RMSE the root of their mean square.

In [ ]:
TARGET_SE = 0.4
MAX_ITEMS = 30
N_EXAMINEES = 500

stopper = StopRule(StandardErrorStopper(TARGET_SE), MaxLengthStopper(MAX_ITEMS))
engine = CatEngine(item_bank, FixedPointInitializer(0.0), MaxInfoSelector(), EAPEstimator(), stopper)
true_thetas = numpy.random.default_rng(POPULATION_SEED).normal(0, 1, N_EXAMINEES)
result = simulate(engine, true_thetas=true_thetas, rng=RUN_SEED)

for name, metric in {
  "bias": result.bias(),
  "RMSE": result.rmse(),
  "test length": result.test_length(),
  "final standard error": result.se(),
}.items():
  print(f"{name}: {metric.value:.3f} ± {metric.mc_se:.3f}")

rates = result.exposure_rates().value
print(f"highest exposure rate {rates.max():.2f}")
print(f"items never given: {numpy.count_nonzero(rates == 0)} of {item_bank.n_items}")
print(f"overlap rate: {result.overlap_rate()}")

The bias, 0.000 ± 0.018, is within its Monte Carlo error of 0. The RMSE, 0.405, is close to the mean
final standard error, 0.395: the standard errors the estimator reports match the errors it makes.
Tests take 18.6 items on average. Almost every test ends just below the 0.4 target, so the mean
final standard error barely varies between examinees, and its Monte Carlo error rounds to 0.

The highest exposure rate is 1: every test starts at the same estimate, so every examinee gets the
same first item. 203 of the 300 items are never given, because maximum information keeps choosing
the most discriminating items. The overlap rate, the share of items two examinees have in common, is
None because the tests differ in length.

## The result's tables

`result.examinees()` has one row per examinee and `result.steps()` one row per item given. Each is a
dict of NumPy columns, which `pandas.DataFrame(result.examinees())` turns into a data frame.
`result.deciles()` gives the metrics by decile of the true ability, from the lowest tenth of the
population to the highest, each metric followed by its Monte Carlo error.

In [ ]:
examinees = result.examinees()
print(list(examinees))
lengths = examinees["test_length"]
print(f"test length: shortest {lengths.min()}, longest {lengths.max()}")
above = examinees["se"] > TARGET_SE
print(f"examinees who ended above the target SE: {numpy.count_nonzero(above)}")
print(f"their true abilities: {numpy.sort(examinees['true_theta'][above]).round(2)}")

deciles = result.deciles()
print("decile  mean θ   RMSE ± mc_se   length")
for decile, theta, rmse, rmse_mc_se, length in zip(
  deciles["decile"],
  deciles["true_theta"],
  deciles["rmse"],
  deciles["rmse_mc_se"],
  deciles["test_length"],
  strict=True,
):
  print(f"{decile:6} {theta:7.2f} {rmse:6.3f} ± {rmse_mc_se:.3f} {length:6.1f}")

Tests take 15 to 30 items. Six examinees ended with a standard error still above the target, so
their tests ran to the 30-item cap; all but one of them have true abilities beyond ±2, where the
bank has few informative items. The deciles show the same from the other side: the tests of the
lowest and highest tenth of the population are the longest, 20.1 and 22.6 items on average against
about 17 in the middle, and their RMSE is the highest, 0.54 and 0.52. With about 50 examinees in
each decile, though, each RMSE carries a Monte Carlo error of about 0.04, so the deciles in between
cannot be ranked.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(examinees["true_theta"], examinees["theta"], s=10, alpha=0.6)
ax.axline((0, 0), slope=1, color="gray", linestyle="--")
ax.set_xlabel("true ability θ")
ax.set_ylabel("final estimate θ̂")
plt.show()

Each point is one examinee. The points scatter around the diagonal, where the estimate equals the
true ability. The errors grow toward the ends of the scale, and there most estimates lie closer to 0
than the true ability: the EAP estimate is pulled toward the mean of its prior, the more so the less
the responses say. The next notebook compares estimators.

## One examinee's test

`result.sessions` holds each examinee's test, in the order of `true_thetas`, and the rows of the
steps table whose `session` is 0 are the first examinee's: the name of each item given, the
response, and the estimate and its standard error after it. `plot.session_progress` draws the
session.

In [ ]:
steps = result.steps()
rows = steps["session"] == 0
for name, response, theta, se in zip(
  steps["name"][rows], steps["response"][rows], steps["theta"][rows], steps["se"][rows], strict=True
):
  print(f"{name}: {'correct' if response else 'incorrect':9} estimate {theta:5.2f}, standard error {se:.2f}")
print(f"true ability {result.true_thetas[0]:.2f}")

plot.session_progress(result.sessions[0], item_bank, true_theta=result.true_thetas[0])
plt.show()

The estimate (blue) starts at 0 and moves most in the first items, when one response changes it a
lot: up after three correct responses, down after two incorrect ones. It then settles, and ends
within one standard error of the true ability (dashed), 0.19. The standard error (green, right axis)
falls with every response, from 0.92 after the first item to 0.39 after the 18th, when the test
stops for reaching the target, and the shaded band of one standard error on each side narrows with
it. The item difficulties (orange) stay near the estimate, most of them a little below it: maximum
information chooses the item whose information peaks at the estimate, and with guessing an item's
peak lies a little above its difficulty.

## Repeating the run

The same seed and examinees repeat every test. A run without `rng` draws fresh random numbers, so
its tests differ every time and cannot be repeated. The
[Reproducibility](https://douglasrizzo.com.br/catsim/reproducibility.html) guide explains what a
seed fixes, and how a study that runs on several processes still repeats.

In [ ]:
again = simulate(engine, true_thetas=true_thetas, rng=RUN_SEED)
unseeded = simulate(engine, true_thetas=true_thetas)
print("same seed, same tests:", again.sessions == result.sessions)
print("no seed, same tests:", unseeded.sessions == result.sessions)

## Where to go next

- [02_comparing_strategies.ipynb](02_comparing_strategies.ipynb) compares estimators, selectors,
  exposure control and stoppers, each difference with its Monte Carlo error.
- The [Running simulations](https://douglasrizzo.com.br/catsim/running-simulations.html) guide
  covers this notebook's tasks as a reference.
- The [technique pages](https://douglasrizzo.com.br/catsim/api/techniques/index.html) define each
  method and say when to choose it.